# MST Inference — V7 Model
## Pipeline
```
Input image(s)
→ MediaPipe skin patch extraction (cheek → forehead → blob)
→ Resize/pad to 224×224
→ ToTensor + ImageNet normalize
→ EfficientNet-B0 (weights)
→ TTA (n=5) — average logits
→ argmax + 1 = predicted MST (1-10)
→ MST name mapping
```
## Usage
- **Single image:** `predict_single(image_path)`
- **Batch:** `predict_batch([path1, path2, ...])`
- **API:** run Cell 5 to start Flask server, POST to `/predict`

# ✅ CELL 1 — Imports

In [ ]:
# Uninstall conflicting packages, reinstall full OpenCV
import subprocess
subprocess.run(["pip", "uninstall", "opencv-python", "opencv-python-headless", "opencv-contrib-python", "-y"])
subprocess.run(["pip", "install", "opencv-python"])

In [ ]:
# !pip install mediapipe opencv-python scikit-image flask

import os
import io
import cv2
import json
import urllib.request
import numpy as np
import torch
import torch.nn as nn
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from torchvision import models, transforms
from PIL import Image
from skimage import color as skcolor
from skimage import measure
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
import cv2
pts = np.array([[0,0],[1,0],[0,1]], dtype=np.int32)
hull = cv2.convexHull(pts)
print("✅ cv2.convexHull works")

# ✅ CELL 2 — Config
Edit paths here only.

In [ ]:
# ── PATHS ──
MODEL_WEIGHTS   = r"D:\Skintone\sc_3\v1\best_v7_model.pth"       # V7 best weights
MEDIAPIPE_MODEL = r"D:\Skintone\sc_3\v1\face_landmarker.task"    # MediaPipe model

# ── MODEL CONFIG — must match V7 training ──
NUM_CLASSES   = 10
TARGET_SIZE   = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
N_TTA         = 5

# ── SKIN LAB RANGE ──
SKIN_L_MIN, SKIN_L_MAX = 20, 90
SKIN_A_MIN, SKIN_A_MAX = 3,  25
SKIN_B_MIN, SKIN_B_MAX = 3,  40
MIN_SKIN_PX             = 500

# ── MST SCALE NAME MAPPING ──
MST_NAMES = {
    1:  {"name": "Monk 1",  "description": "Very Light",        "hex": "#f6ede4"},
    2:  {"name": "Monk 2",  "description": "Light",             "hex": "#f3e7db"},
    3:  {"name": "Monk 3",  "description": "Light Medium",      "hex": "#f7ead0"},
    4:  {"name": "Monk 4",  "description": "Medium Light",      "hex": "#eadaba"},
    5:  {"name": "Monk 5",  "description": "Medium",            "hex": "#d7bd96"},
    6:  {"name": "Monk 6",  "description": "Medium Dark",       "hex": "#a07850"},
    7:  {"name": "Monk 7",  "description": "Dark Medium",       "hex": "#825c43"},
    8:  {"name": "Monk 8",  "description": "Dark",              "hex": "#604134"},
    9:  {"name": "Monk 9",  "description": "Very Dark",         "hex": "#3a312a"},
    10: {"name": "Monk 10", "description": "Deepest",           "hex": "#292420"},
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
assert os.path.exists(MODEL_WEIGHTS), f"❌ Weights not found: {MODEL_WEIGHTS}"

# ✅ CELL 3 — Load Model + MediaPipe + Preprocessing
All inference logic defined here. Run once.

In [ ]:
# ── Load MediaPipe ──
if not os.path.exists(MEDIAPIPE_MODEL):
    print("Downloading MediaPipe model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        MEDIAPIPE_MODEL
    )

mp_base    = python.BaseOptions(model_asset_path=MEDIAPIPE_MODEL)
mp_options = vision.FaceLandmarkerOptions(base_options=mp_base, num_faces=1)
landmarker = vision.FaceLandmarker.create_from_options(mp_options)

LEFT_CHEEK_IDX  = [116, 123, 147, 213, 192, 214, 210, 211, 32, 208, 206, 203]
RIGHT_CHEEK_IDX = [345, 352, 376, 433, 416, 434, 430, 431, 262, 428, 426, 423]
FOREHEAD_IDX    = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
                   67, 109, 10, 151, 9, 8, 107, 66, 105, 63, 70]
NOSE_TIP=1; LEFT_EAR=234; RIGHT_EAR=454


def get_landmarks(path):
    try:
        result = landmarker.detect(mp.Image.create_from_file(str(path)))
        if result.face_landmarks: return result.face_landmarks[0]
    except: pass
    return None

def face_orientation(lm):
    n=lm[NOSE_TIP]; l=lm[LEFT_EAR]; r=lm[RIGHT_EAR]
    dl=abs(n.x-l.x); dr=abs(n.x-r.x); total=dl+dr
    if total<1e-6: return 'frontal'
    if abs(dr-dl)/total<0.15: return 'frontal'
    return 'right' if dl<dr else 'left'

def lm_pts(lm, idx, h, w):
    return np.array([[int(lm[i].x*w),int(lm[i].y*h)] for i in idx if i<len(lm)],dtype=np.int32)

def make_mask(pts, h, w):
    if len(pts)<3: return None
    hull=cv2.convexHull(pts); mask=np.zeros((h,w),dtype=np.uint8)
    cv2.fillConvexPoly(mask,hull,255); return mask

def skin_filter(img_np, region_mask):
    lab=skcolor.rgb2lab(img_np); L,A,B=lab[:,:,0],lab[:,:,1],lab[:,:,2]
    skin=((L>=SKIN_L_MIN)&(L<=SKIN_L_MAX)&(A>=SKIN_A_MIN)&(A<=SKIN_A_MAX)&(B>=SKIN_B_MIN)&(B<=SKIN_B_MAX))
    combined=(region_mask>0)&skin; out=img_np.copy(); out[~combined]=0.0
    return out, combined

def to_patch(masked, valid):
    if valid.sum()<MIN_SKIN_PX: return None
    rows=np.any(valid,axis=1); cols=np.any(valid,axis=0)
    r0,r1=np.where(rows)[0][[0,-1]]; c0,c1=np.where(cols)[0][[0,-1]]
    crop=masked[r0:r1+1,c0:c1+1]; ch,cw=crop.shape[:2]
    pil=Image.fromarray((crop*255).astype(np.uint8))
    if ch>=TARGET_SIZE and cw>=TARGET_SIZE:
        return pil.resize((TARGET_SIZE,TARGET_SIZE),Image.BILINEAR)
    canvas=Image.new("RGB",(TARGET_SIZE,TARGET_SIZE),(0,0,0))
    canvas.paste(pil,((TARGET_SIZE-min(cw,TARGET_SIZE))//2,(TARGET_SIZE-min(ch,TARGET_SIZE))//2))
    return canvas

def extract_patch(image_path):
    """Extract skin patch from image. Returns (PIL 224x224, method) or (None, None)."""
    try:
        img=Image.open(str(image_path)).convert("RGB").resize((512,512),Image.BILINEAR)
        img_np=np.array(img,dtype=np.float32)/255.0
    except Exception as e:
        return None, f"load_error: {e}"
    h,w=img_np.shape[:2]
    lm=get_landmarks(str(image_path))
    if lm is not None:
        orient=face_orientation(lm)
        if orient=='frontal':
            lmsk=make_mask(lm_pts(lm,LEFT_CHEEK_IDX,h,w),h,w)
            rmsk=make_mask(lm_pts(lm,RIGHT_CHEEK_IDX,h,w),h,w)
            if lmsk is not None and rmsk is not None:
                m,s=skin_filter(img_np,np.maximum(lmsk,rmsk))
                p=to_patch(m,s)
                if p: return p,'both_cheeks'
        order=[RIGHT_CHEEK_IDX,LEFT_CHEEK_IDX] if orient=='left' else [LEFT_CHEEK_IDX,RIGHT_CHEEK_IDX]
        for cidx in order:
            msk=make_mask(lm_pts(lm,cidx,h,w),h,w)
            if msk is not None:
                m,s=skin_filter(img_np,msk); p=to_patch(m,s)
                if p: return p,'single_cheek'
        fmsk=make_mask(lm_pts(lm,FOREHEAD_IDX,h,w),h,w)
        if fmsk is not None:
            m,s=skin_filter(img_np,fmsk); p=to_patch(m,s)
            if p: return p,'forehead'
    lab=skcolor.rgb2lab(img_np); L,A,B=lab[:,:,0],lab[:,:,1],lab[:,:,2]
    skin=((L>=SKIN_L_MIN)&(L<=SKIN_L_MAX)&(A>=SKIN_A_MIN)&(A<=SKIN_A_MAX)&(B>=SKIN_B_MIN)&(B<=SKIN_B_MAX))
    if skin.sum()<MIN_SKIN_PX: return None,'no_skin_found'
    labeled=measure.label(skin); regions=measure.regionprops(labeled)
    if not regions: return None,'no_region'
    lg=max(regions,key=lambda r:r.area); bm=(labeled==lg.label)
    masked=img_np.copy(); masked[~bm]=0.0
    p=to_patch(masked,bm)
    return (p,'blob_fallback') if p else (None,'blob_too_small')


# ── Load Model ──
def load_model(weights_path):
    m = models.efficientnet_b0(weights=None)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.5, inplace=True),
        nn.Linear(in_features, NUM_CLASSES)
    )
    m.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    m = m.to(device)
    m.eval()
    return m

model = load_model(MODEL_WEIGHTS)
print(f"✅ Model loaded from: {MODEL_WEIGHTS}")


# ── Transforms — +
val_tf = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
tta_tf = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.0, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f"✅ Preprocessing ready. TTA n={N_TTA}")

# ✅ CELL 4 — Inference Functions
- `predict_single(image_path)` — one image, returns dict
- `predict_batch(image_paths)` — list of images, returns list of dicts
- `predict_from_pil(pil_image)` — PIL image input (for API use)

In [ ]:
def predict_from_patch(patch):
    """
    Run TTA inference on an already-extracted 224x224 PIL patch.
    Returns predicted MST (1-10) and confidence scores.
    """
    logits_sum = None
    with torch.no_grad():
        for i in range(N_TTA):
            t   = val_tf(patch) if i == 0 else tta_tf(patch)
            out = model(t.unsqueeze(0).to(device))
            logits_sum = out if logits_sum is None else logits_sum + out

    probs     = torch.softmax(logits_sum, dim=1).squeeze().cpu().numpy()
    pred_idx  = int(probs.argmax())
    mst       = pred_idx + 1  # 0-indexed → 1-indexed
    confidence = float(probs[pred_idx])

    # Top 3 predictions
    top3_idx  = probs.argsort()[::-1][:3]
    top3      = [{"mst": int(i)+1, "name": MST_NAMES[int(i)+1]["name"],
                  "probability": round(float(probs[i]), 4)} for i in top3_idx]

    return mst, confidence, top3, probs.tolist()


def predict_single(image_path):
    """
    Predict MST for a single image file.

    Args:
        image_path: str or Path to image file

    Returns dict:
        {
            "file":          filename,
            "mst":           predicted MST (1-10),
            "name":          e.g. "Monk 4",
            "description":   e.g. "Medium Light",
            "hex":           e.g. "#eadaba",
            "confidence":    float 0-1,
            "top3":          [{mst, name, probability}, ...],
            "extraction":    method used (both_cheeks/single_cheek/forehead/blob_fallback),
            "status":        "success" or "failed",
            "error":         error message if failed
        }
    """
    image_path = Path(image_path)
    result = {"file": image_path.name, "status": "failed", "error": None}

    patch, method = extract_patch(image_path)

    if patch is None:
        result["error"] = method  # method contains error reason when None
        return result

    mst, confidence, top3, all_probs = predict_from_patch(patch)
    mst_info = MST_NAMES[mst]

    result.update({
        "status":      "success",
        "mst":         mst,
        "name":        mst_info["name"],
        "description": mst_info["description"],
        "hex":         mst_info["hex"],
        "confidence":  round(confidence, 4),
        "top3":        top3,
        "extraction":  method,
        "all_probs":   [round(p, 4) for p in all_probs],
    })
    return result


def predict_batch(image_paths, verbose=True):
    """
    Predict MST for a list of image paths.

    Args:
        image_paths: list of str or Path
        verbose:     show progress bar

    Returns: list of result dicts (same format as predict_single)
    """
    results = []
    iterator = tqdm(image_paths, desc="Predicting") if verbose else image_paths
    for path in iterator:
        results.append(predict_single(path))
    return results


def predict_from_pil(pil_image, filename="image"):
    """
    Predict MST from a PIL Image object.
    Useful when image comes from API request bytes.

    Args:
        pil_image: PIL.Image
        filename:  name to include in result

    Returns: result dict
    """
    import tempfile
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        pil_image.save(tmp.name, "JPEG", quality=95)
        result = predict_single(tmp.name)
        result["file"] = filename
    os.unlink(tmp.name)
    return result


def print_result(result):
    """Pretty print a single prediction result."""
    if result["status"] == "failed":
        print(f"❌ {result['file']} — Failed: {result['error']}")
        return
    print(f"\n{'='*45}")
    print(f"File       : {result['file']}")
    print(f"MST        : {result['mst']} — {result['name']}")
    print(f"Description: {result['description']}")
    print(f"Hex color  : {result['hex']}")
    print(f"Confidence : {result['confidence']:.1%}")
    print(f"Extraction : {result['extraction']}")
    print(f"Top 3:")
    for t in result['top3']:
        print(f"  MST {t['mst']:>2} ({t['name']:<10}): {t['probability']:.1%}")
    print(f"{'='*45}")


print("✅ Inference functions ready.")
print("   predict_single(path)         — single image")
print("   predict_batch([path1, ...])  — multiple images")
print("   predict_from_pil(pil_image)  — PIL input (API)")

# ✅ CELL 5 — Run Inference
Edit the paths below. Supports single image, folder, or list.

In [ ]:
# ── Single image ──
# result = predict_single(r"D:\path\to\your\image.jpg")
# print_result(result)


# ── Multiple images ──
image_paths = [
    r"D:\Skintone\sc_3\v1\example\exp1.jpg",
    r"D:\Skintone\sc_3\v1\example\exp2.png",
    r"D:\Skintone\sc_3\v1\example\exp3.png",
    r"D:\Skintone\sc_3\v1\example\exp4.png",
    r"D:\Skintone\sc_3\v1\example\exp5.png",
    r"D:\Skintone\sc_3\v1\example\exp6.png"
]
results = predict_batch(image_paths)
for r in results:
    print_result(r)


# ── Entire folder ──
# folder = r"D:\path\to\folder"
# image_paths = [str(p) for p in Path(folder).glob("*.jpg")]
# results = predict_batch(image_paths)
# for r in results:
#     print_result(r)


# ── Export results to JSON ──
# import json
# with open("predictions.json", "w") as f:
#     json.dump(results, f, indent=2)
# print("Saved: predictions.json")

# ✅ CELL 6 — Flask API Server
Run this cell to start the API server.

**Endpoints:**
- `POST /predict` — single image upload
- `POST /predict/batch` — multiple image uploads
- `GET /health` — server health check

**Example call from web app:**
```javascript
const formData = new FormData();
formData.append('image', fileInput.files[0]);
const response = await fetch('http://localhost:5000/predict', {
    method: 'POST',
    body: formData
});
const result = await response.json();
console.log(result.mst, result.name, result.hex);
```

In [ ]:
from flask import Flask, request, jsonify
from PIL import Image
import io

app = Flask(__name__)

API_PORT = 5000


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        "status": "ok",
        "model":  "V7 EfficientNet-B0",
        "device": str(device),
        "tta":    N_TTA
    })


@app.route('/predict', methods=['POST'])
def predict_api():
    """
    Single image prediction.
    Accepts: multipart/form-data with 'image' field
    Returns: JSON result dict
    """
    if 'image' not in request.files:
        return jsonify({"error": "No image field in request"}), 400

    file = request.files['image']
    if file.filename == '':
        return jsonify({"error": "Empty filename"}), 400

    try:
        pil_image = Image.open(io.BytesIO(file.read())).convert("RGB")
        result    = predict_from_pil(pil_image, filename=file.filename)

        if result["status"] == "failed":
            return jsonify(result), 422

        return jsonify(result), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route('/predict/batch', methods=['POST'])
def predict_batch_api():
    """
    Batch image prediction.
    Accepts: multipart/form-data with multiple 'images' fields
    Returns: JSON list of result dicts
    """
    if 'images' not in request.files:
        return jsonify({"error": "No images field in request"}), 400

    files   = request.files.getlist('images')
    results = []

    for file in files:
        try:
            pil_image = Image.open(io.BytesIO(file.read())).convert("RGB")
            result    = predict_from_pil(pil_image, filename=file.filename)
            results.append(result)
        except Exception as e:
            results.append({
                "file": file.filename,
                "status": "failed",
                "error": str(e)
            })

    summary = {
        "total":   len(results),
        "success": sum(1 for r in results if r["status"] == "success"),
        "failed":  sum(1 for r in results if r["status"] == "failed"),
        "results": results
    }
    return jsonify(summary), 200


print(f"🚀 Starting API server on http://localhost:{API_PORT}")
print(f"   POST http://localhost:{API_PORT}/predict         — single image")
print(f"   POST http://localhost:{API_PORT}/predict/batch   — multiple images")
print(f"   GET  http://localhost:{API_PORT}/health          — health check")
print(f"\n   Press Ctrl+C to stop.")

app.run(host='0.0.0.0', port=API_PORT, debug=False, use_reloader=False)